# BIOT 6900 · Module 2 — Multi-Omics Target Identification & Validation
### Week 2 · Computational Lab 2 · Starter Notebook

**What this notebook does.** You'll integrate three data layers — transcriptomics (RNA), proteomics (protein), and genomics — to nominate and *rank* disease targets, then hand that ranked list to Weeks 3–4.

**How today is built — I do → we do → you do:**
- **Part 1 (worked, already complete):** CPTAC **breast cancer** on *synthetic* data. The tumors are measured at all three layers, so the data is *sample-matched* — you integrate **at the sample level**. Your instructor runs these cells; follow along.
- **Part 2 (together, in class):** find and load *real* CPTAC data, then re-run Part 1 on real numbers. Not graded.
- **Part 3 (your assignment, `# TODO` cells):** **Alzheimer's disease**. The cohorts are *not* matched across layers, so you integrate **at the gene level**. You transfer the Part 1 method to messier, more realistic data.

> **The one idea to carry through:** *how* you integrate is dictated by *what your data has matched.* Matched → correlate within samples. Unmatched → compare gene-level summaries. Same goal, different machinery.

**Data.** Part 1 looks for the synthetic CPTAC files in `data/` and, if they're absent, falls back to clearly-labelled demo data so it always runs. Part 3 requires the three Alzheimer's matrices posted on **Canvas** — place them in `data/` (see the Lab Guide).

*Continuity from Module 1:* you'll see **TP53 / p53** (`P04637`, PDB `1TUP`) surface in the cancer half and **APOE** (`rs7412`) in the Alzheimer's half.

**Before you submit:** `Kernel → Restart & Run All` so the whole notebook executes top to bottom.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(6900)  # fixed seed so results are reproducible

# Module 1 continuity anchors
P53_UNIPROT, P53_PDB = "P04637", "1TUP"   # p53  -> cancer half
APOE_VARIANT = "rs7412"                    # APOE -> Alzheimer's half

# Default scoring weights for this course: equal across the three layers.
# You MAY change these in Part 3 — but if you do, you must justify it in your report.
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

## Helper functions you'll use in both parts
Two small functions do the scoring work. `rank_percentile` puts any layer's scores on a common 0–1 scale (by rank, so it's robust to outliers and to layers being on different units). `multi_evidence_score` normalizes each layer and takes the weighted sum.

In [2]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## Data loading (provided — you don't need to edit these)
`load_cptac()` reads the matched breast-cancer matrices for Part 1 (or synthesizes demo data if they're missing). `load_ad()` reads the three Alzheimer's matrices you download from Canvas for Part 2.

In [3]:
CPTAC_GENES = ["TP53", "PIK3CA", "ERBB2", "ESR1", "GATA3", "MYC", "CDH1",
               "MAP3K1", "PTEN", "AKT1", "RB1", "CCND1", "FOXA1", "MKI67",
               "EGFR", "BRCA1", "BRCA2", "KRT5", "VIM", "ACTB"]


def _synth_cptac(n_tumor=60, n_normal=15):
    genes = CPTAC_GENES
    tumor_ids = [f"T{i:02d}" for i in range(n_tumor)]
    normal_ids = [f"N{i:02d}" for i in range(n_normal)]
    coupling = pd.Series(RNG.uniform(0.05, 0.80, len(genes)), index=genes)

    def gene_matrix(sample_ids, shift):
        rna = pd.DataFrame(RNG.normal(0, 1, (len(genes), len(sample_ids))),
                           index=genes, columns=sample_ids).add(shift, axis=0)
        noise = pd.DataFrame(RNG.normal(0, 1, rna.shape), index=genes, columns=sample_ids)
        prot = rna.mul(coupling, axis=0) + noise.mul(1 - coupling + 0.35, axis=0)
        return rna, prot

    shift = pd.Series(0.0, index=genes)
    for g in ["TP53", "ERBB2", "MKI67", "MYC", "PIK3CA"]:
        shift[g] = RNG.uniform(1.2, 2.2)
    rna_t, prot_t = gene_matrix(tumor_ids, shift)
    rna_n, prot_n = gene_matrix(normal_ids, pd.Series(0.0, index=genes))
    rna = pd.concat([rna_t, rna_n], axis=1)
    prot = pd.concat([prot_t, prot_n], axis=1)
    mut_freq = pd.Series(RNG.uniform(0.0, 0.05, len(genes)), index=genes)
    for g in ["TP53", "PIK3CA", "CDH1", "GATA3"]:
        mut_freq[g] = RNG.uniform(0.25, 0.45)
    meta = pd.Series(["tumor"] * n_tumor + ["normal"] * n_normal,
                     index=tumor_ids + normal_ids, name="group")
    return rna, prot, mut_freq, meta


def load_cptac():
    p = {"rna": "data/cptac_brca_rna.tsv", "prot": "data/cptac_brca_protein.tsv",
         "mut": "data/cptac_brca_mutation.tsv"}
    if all(os.path.exists(v) for v in p.values()):
        rna = pd.read_csv(p["rna"], sep="\t", index_col=0)
        prot = pd.read_csv(p["prot"], sep="\t", index_col=0)
        mut = pd.read_csv(p["mut"], sep="\t", index_col=0).iloc[:, 0]
        print("Loaded CPTAC files from data/.")
        return rna, prot, mut, None
    print("!! WARNING: CPTAC files not found -> SYNTHETIC demo data "
          "(illustrative only, not real CPTAC values).")
    return _synth_cptac()


def load_ad():
    p = {"tx": "data/ad_transcriptomics.tsv", "pr": "data/ad_proteomics.tsv",
         "gw": "data/ad_gwas.tsv"}
    missing = [v for v in p.values() if not os.path.exists(v)]
    if missing:
        raise FileNotFoundError(
            "Alzheimer's matrices not found: " + ", ".join(missing) +
            "\nDownload the three files from Canvas and put them in a 'data/' folder "
            "next to this notebook (see the Lab Guide).")
    return (pd.read_csv(p["tx"], sep="\t"),
            pd.read_csv(p["pr"], sep="\t"),
            pd.read_csv(p["gw"], sep="\t"))

---
# Part 1 — Worked example: CPTAC breast cancer *(run and read; nothing to edit)*

**Why integrate at all?** Each layer can produce artifacts the others don't share, so a target supported across genome, transcriptome, and proteome is a stronger bet than one seen in only one layer. Integration is triangulation.

Because CPTAC measures the **same tumors** at every layer, we can line samples up and integrate **at the sample level.**

In [4]:
rna, prot, mut_freq, meta = load_cptac()
print("RNA matrix:    ", rna.shape, "(genes x samples)")
print("Protein matrix:", prot.shape)
print("Mutation freq: ", mut_freq.shape)

Loaded CPTAC files from data/.
RNA matrix:     (800, 100) (genes x samples)
Protein matrix: (500, 100)
Mutation freq:  (800,)


### 1.2 Harmonize identifiers → a common gene key
The real technical wall in multi-omics: gene symbols, Ensembl IDs, and UniProt accessions don't line up for free. Here we intersect on a shared key and check how many genes *survive the join* — always your first reality check.

In [5]:
common = rna.index.intersection(prot.index).intersection(mut_freq.index)
print(f"Genes surviving the 3-way join: {len(common)} "
      f"(RNA {len(rna.index)}, protein {len(prot.index)}, mutation {len(mut_freq.index)})")
rna, prot, mut_freq = rna.loc[common], prot.loc[common], mut_freq.loc[common]
samples = rna.columns.intersection(prot.columns)

Genes surviving the 3-way join: 500 (RNA 800, protein 500, mutation 800)


### 1.3–1.4 Sample-level RNA–protein correlation
With matched samples we can ask, per gene, *does protein track RNA across patients?* The answer is usually **partial** — correlation is modest and varies by gene. A gene where protein does **not** track RNA isn't broken data; it's a signal of post-transcriptional regulation (translational buffering, protein turnover).

In [6]:
corr = pd.Series(
    {g: stats.spearmanr(rna.loc[g, samples], prot.loc[g, samples]).statistic
     for g in common}, name="rna_prot_corr")
print(f"RNA-protein correlation: median={corr.median():.2f}, "
      f"range=[{corr.min():.2f}, {corr.max():.2f}]   <- note: NOT ~1.0")
print(f"  most coupled: {corr.idxmax()} ({corr.max():.2f});  "
      f"most buffered: {corr.idxmin()} ({corr.min():.2f})")

RNA-protein correlation: median=0.38, range=[-0.17, 0.87]   <- note: NOT ~1.0
  most coupled: BG0208 (0.87);  most buffered: BG0285 (-0.17)


**Read this result.** The median correlation is well below 1 — most genes' protein levels only partly follow their RNA. That is the empirical reason you can't treat RNA as a stand-in for protein, and why integrating both layers adds information.

### 1.5 Per-layer differential signal
Each layer independently nominates candidates. Here the signal is the tumor-vs-normal effect size at RNA and protein, plus per-gene mutation frequency for the genomic layer.

In [7]:
if meta is not None:
    tcols = meta.index[meta == "tumor"]
    ncols = meta.index[meta == "normal"]
    rna_eff = (rna[tcols].mean(axis=1) - rna[ncols].mean(axis=1)).abs()
    prot_eff = (prot[tcols].mean(axis=1) - prot[ncols].mean(axis=1)).abs()
else:
    rna_eff = rna[samples].mean(axis=1).abs()
    prot_eff = prot[samples].mean(axis=1).abs()

scored = pd.DataFrame({"transcriptomic": rna_eff, "proteomic": prot_eff,
                       "genomic": mut_freq, "rna_prot_corr": corr})
scored.round(3).head()

,transcriptomic,proteomic,genomic,rna_prot_corr
TP53,2.093,0.978,0.413,0.531
PIK3CA,1.524,0.131,0.369,0.029
ERBB2,1.481,0.353,0.050,0.177
ESR1,1.362,0.408,0.028,0.173
GATA3,2.237,1.297,0.424,0.525


### 1.6 Multi-evidence score → ranked targets
Normalize each layer to a common scale and take the **equal-weighted** sum. This turns three noisy layers into one ranked list — the artifact everything downstream consumes.

In [8]:
scored["score"] = multi_evidence_score(
    scored, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
ranked_cptac = scored.sort_values("score", ascending=False)
print("Top CPTAC targets (worked example):")
print(ranked_cptac.head(6).round(3).to_string())
# Continuity check: TP53 / p53 (Module 1's P04637 / 1TUP) is a top breast-cancer hit.
print("\nTP53 rank:", list(ranked_cptac.index).index("TP53") + 1)

Top CPTAC targets (worked example):
        transcriptomic  proteomic  genomic  rna_prot_corr  score
GATA3            2.237      1.297    0.424          0.525  0.997
TP53             2.093      0.978    0.413          0.531  0.990
MAP3K1           1.572      1.182    0.290          0.789  0.987
CDH1             1.622      0.472    0.443         -0.135  0.985
ERBB2            1.481      0.353    0.050          0.177  0.975
BG0050           0.427      0.400    0.045          0.727  0.947

TP53 rank: 2


### Part 1 recap
You integrated **at the sample level** because the data was **matched** — you could correlate RNA and protein within the same tumors. That was synthetic data, so the method is easy to see. Next we run it on real data, then you transfer it to unmatched data yourself.

---
# Part 2 — Find and load real CPTAC data *(together, in class)*

Now swap the synthetic files for the real thing. CPTAC breast data is open access (no data-use agreement) — we'll get it together in class. **Not graded.**

1. Go to **LinkedOmics** (`linkedomics.org`) → **CPTAC Breast Cancer (BRCA)**.
2. Download the gene-level **RNAseq** and **Proteome** matrices (gene × sample).
3. Save them into `data/` with genes as the row index and these exact names — transpose with `.T` first if samples are in rows:
   - `data/cptac_brca_rna.tsv`
   - `data/cptac_brca_protein.tsv`
   - `data/cptac_brca_mutation.tsv`  (one row per gene: mutation frequency or CNV magnitude)
4. `Kernel → Restart & Run All` and watch **Part 1** again — it now loads your real files instead of the demo. Compare: is the RNA–protein correlation still modest? Does TP53 still surface near the top?

*Wrinkle:* the real download has no tumor/normal labels, so ranking falls back to overall abundance rather than a tumor-vs-normal effect. That's fine for seeing the pipeline run on real data.

---
# Part 3 — Your assignment: Ovarian Cancer *(complete the `# TODO` cells)*

The three matrices you downloaded from Canvas are **gene-level summaries** from different cohorts — there are no shared samples to correlate, so you integrate **at the gene level.** You'll transfer the Part 1 workflow: harmonize → concordance → score → rank → export.

Your ranked target list (the CSV you export in 3.5) is what **Week 3** builds on, so it has to be clean. *Continuity:* expect **APOE** (`rs7412`) among your top hits.

### 3.1 — TODO: load and inspect the three matrices
Call `load_ad()` and look at each table's columns and shape before you touch them.

In [ ]:
# TODO 3.1 — load the three Ovarian Cancer's matrices and inspect them.
def load_ov():
    p = {"rna": "data/ov_rna.cct", "prot": "data/ov_protein.cct.txt", "mut": "data/ov_mutation.cbt.txt"}
    rna = pd.read_csv(p["rna"], sep="\t", index_col=0)
    prot = pd.read_csv(p["prot"], sep="\t", index_col=0)
    mut = pd.read_csv(p["mut"], sep="\t", index_col=0)
    return rna, prot, mut

rna, prot, mut = load_ov()
print("RNA matrix:     ", rna.shape, "(genes x samples)")
print("Protein matrix: ", prot.shape)
print("Mutation matrix:", mut.shape, "(binary, genes x samples)")

RNA matrix:      (20032, 303) (genes x samples)
Protein matrix:  (6475, 84)
Mutation matrix: (2205, 465) (binary, genes x samples)


### 3.2 — TODO: harmonize on the gene symbol → one joined table
Rename the effect/`pval` columns so RNA and protein don't collide, then merge all three on `gene`. Report how many genes survive the join (your first reality check).

In [ ]:
# TODO 3.2 — build a single joined table `df`.
# Collapse the binary mutation matrix to a per-gene mutation frequency (fraction of tumors mutated)
mut_freq = mut.astype(float).mean(axis=1)
mut_freq.name = "mut_freq"

common = rna.index.intersection(prot.index).intersection(mut_freq.index)
print(f"Genes surviving the 3-way join: {len(common)} "
      f"(RNA {len(rna.index)}, protein {len(prot.index)}, mutation {len(mut_freq.index)})")

rna, prot, mut_freq = rna.loc[common], prot.loc[common], mut_freq.loc[common]
samples = rna.columns.intersection(prot.columns)
print(f"Samples with BOTH RNA and protein measured (matched, sample-level): {len(samples)}")

Genes surviving the 3-way join: 751 (RNA 20032, protein 6475, mutation 2205)
Samples with BOTH RNA and protein measured (matched, sample-level): 54


### 3.3 — TODO: sign-agreement concordance
You can't correlate across samples here (nothing is matched), so concordance becomes **direction agreement**: does the gene move the *same way* at RNA and protein? Add a boolean `concordant` column. Watch for genes that are strong at RNA but flat/opposite at protein — those are the interesting discordant ones.

In [ ]:
# TODO 3.3 — add a boolean `concordant` column: do RNA and protein point the same direction?
corr = pd.Series(
    {g: stats.spearmanr(rna.loc[g, samples], prot.loc[g, samples]).statistic
     for g in common}, name="rna_prot_corr")
print(f"RNA-protein correlation: median={corr.median():.2f}, "
      f"range=[{corr.min():.2f}, {corr.max():.2f}]   <- note: NOT ~1.0")
print(f"  most coupled: {corr.idxmax()} ({corr.max():.2f});  "
      f"most buffered: {corr.idxmin()} ({corr.min():.2f})")

RNA-protein correlation: median=0.49, range=[-0.18, 0.88]   <- note: NOT ~1.0
  most coupled: CDH6 (0.88);  most buffered: CPN2 (-0.18)


### 3.4 — TODO: multi-evidence score
Build a per-layer magnitude for each of the three layers, then score with `multi_evidence_score` and `EQUAL_WEIGHTS`. If you change the weights, justify it in your report.

In [ ]:
# TODO 3.4 — score every gene.
rna_eff = rna[samples].mean(axis=1).abs()
prot_eff = prot[samples].mean(axis=1).abs()

scored = pd.DataFrame({"transcriptomic": rna_eff, "proteomic": prot_eff,
                       "genomic": mut_freq, "rna_prot_corr": corr})
scored.round(3).head()

,transcriptomic,proteomic,genomic,rna_prot_corr
AASS,6.851,0.033,0.006,NaN
ABCA1,10.013,0.112,0.013,NaN
ABCC1,10.459,0.052,0.006,NaN
ABCF1,10.972,0.041,0.006,0.688
ACAD10,9.249,0.034,0.006,0.423


### 3.5 — TODO: rank, inspect, and export
Sort by score, take the top ~15, and **export the ranked table to `targets_ad.csv`** — this file is the hand-off to Week 3. Check whether known AD genes (APOE, TREM2, BIN1, CLU, PICALM …) are recovered, and look at any `concordant == False` genes in your top hits.

In [ ]:
# TODO 3.5 — rank, look at the top 15, and export the CSV.
scored["score"] = multi_evidence_score(
    scored, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
ranked = scored.sort_values("score", ascending=False)

top15 = ranked.head(15)
print("Top 15 ovarian cancer targets:")
print(top15.round(3).to_string())

ranked.to_csv("targets_ov.csv")
print(f"\nExported {len(ranked)} ranked genes to targets_ov.csv")

Top 15 ovarian cancer targets:
        transcriptomic  proteomic  genomic  rna_prot_corr  score
MUC16           13.564      0.476    0.043          0.664  0.988
COL6A3          12.327      0.196    0.030          0.343  0.951
FN1             14.175      0.409    0.011          0.714  0.913
MACF1           11.894      0.125    0.030          0.428  0.904
COL3A1          14.059      0.259    0.011          0.458  0.904
CHD4            12.674      0.107    0.015          0.088  0.889
MXRA5           11.579      0.306    0.013          0.613  0.888
KDM5C           11.520      0.149    0.017            NaN  0.884
TP53            11.023      0.151    0.828            NaN  0.875
PRKDC           12.189      0.099    0.015          0.295  0.867
COL1A2          14.837      0.389    0.009          0.297  0.858
NOTCH2          12.648      0.116    0.011            NaN  0.847
MYH9            14.285      0.087    0.011          0.661  0.829
BPTF            10.742      0.165    0.013          0.237  

### 3.6 — Interpretation (write-up, goes in your report)
Answer these in your 3–4 page report — this is the 40% interpretation payload:

1. **Weighting.** You used equal weights. Argue for keeping them equal *or* for up-weighting a layer (e.g. GWAS as germline/causal-leaning vs. transcriptomics as possibly downstream). There's no single right answer — only reasoned vs. unreasoned.
2. **Top targets.** Which known AD genes did you recover (APOE, TREM2, BIN1, CLU, PICALM …)? Any non-obvious hit worth a second look?
3. **Read a discordant gene.** Pick a `concordant == False` gene in your top hits (strong RNA/GWAS, flat protein). What biology could explain RNA and protein disagreeing?
4. **Limitation.** This was **gene-level, cross-cohort** integration — unmatched — so you *cannot* make per-patient claims. Contrast this with the matched CPTAC case from Part 1.

---
### Submit (Part 3 only)
1. `Kernel → Restart & Run All` — confirm the whole notebook runs top to bottom.
2. Commit **this notebook**, **`targets_ad.csv`**, and a short **README** (your name + anything that didn't work) to your `biot6900` repo.
3. Push, confirm the files appear on github.com, then **post your repo link on Canvas.**

Grading follows the course 60 / 40 split — 60 execution, 40 interpretation & communication. Partial credit for a correct approach even with minor technical errors: if a step wouldn't run, say what you were trying to do and what happened.